In [1]:
import importlib.util
import importlib
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [2]:
# Load TucoDataset from compiled .pyc
_pyc_dir = os.path.abspath(os.path.join('..', '__pycache__'))

def _load_pyc(name):
    pyc = os.path.join(_pyc_dir, f'{name}.cpython-310.pyc')
    spec = importlib.util.spec_from_file_location(name, pyc)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    sys.modules[name] = mod

_load_pyc('tuco_dataset')

from tuco_dataset import TucoDataset

DOWNSAMPLE = True
DATA_ROOT = '../../Data/Tuco'
MAX_TRAIN = 100
MAX_VAL = None
WINDOW_RADIUS = 3
N_STACK = 2 * WINDOW_RADIUS + 1

def _load_volumes(split, max_n):
    ds = TucoDataset(DATA_ROOT, split=split)
    if max_n is not None:
        from torch.utils.data import Subset
        ds = Subset(ds, range(min(max_n, len(ds))))
    mr_list, ct_list, mr_seg_list, ct_seg_list = [], [], [], []
    for sample in ds:
        mr_list.append(sample['moving'].squeeze(0).numpy().astype(np.float32))
        ct_list.append(sample['fixed'].squeeze(0).numpy().astype(np.float32))
        mr_seg_list.append(sample['moving_seg'].squeeze(0).numpy().astype(np.int16))
        ct_seg_list.append(sample['fixed_seg'].squeeze(0).numpy().astype(np.int16))
    return mr_list, ct_list, mr_seg_list, ct_seg_list

print('Loading train volumes...')
train_mr_vols, train_ct_vols, train_mr_segs, train_ct_segs = _load_volumes('train', MAX_TRAIN)
print('Loading val volumes...')
val_mr_vols, val_ct_vols, val_mr_segs, val_ct_segs = _load_volumes('val', MAX_VAL)

if DOWNSAMPLE:
    ORIENT_CONFIG = [
        (0, 96,  'axial',    (112, 96)),
        (1, 112, 'coronal',  (96,  96)),
        (2, 96,  'sagittal', (96,  112)),
    ]
else:
    ORIENT_CONFIG = [
        (0, 176, 'axial',    (208, 192)),
        (1, 208, 'coronal',  (176, 192)),
        (2, 192, 'sagittal', (176, 208)),
    ]

SEG_LABELS = [2, 3, 4, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 26, 28]
N_LABELS = len(SEG_LABELS)

n_train = len(train_mr_vols)
stacks_per_subj = sum(d - 2*WINDOW_RADIUS for _, d, _, _ in ORIENT_CONFIG)
print(f'Train: {n_train} subjects x {stacks_per_subj} stacks/subj = {n_train * stacks_per_subj} total')
print(f'Val: {len(val_mr_vols)} subjects')
print(f'{N_LABELS} anatomical labels')

Loading train volumes...
Found 144 volume pairs for split 'train'
Loading val volumes...
Found 18 volume pairs for split 'val'
Train: 100 subjects x 286 stacks/subj = 28600 total
Val: 18 subjects
16 anatomical labels


In [3]:
def multi_orient_generator(mr_vols, ct_vols, mr_segs, ct_segs,
                           batch_size=8, window_radius=WINDOW_RADIUS):
    # Cross-subject, multi-orientation slice-stack generator
    n_subj = len(mr_vols)
    wr = window_radius
    n_stack = 2 * wr + 1

    def _extract_slice(vol, axis, z):
        if axis == 0: return vol[z]
        elif axis == 1: return vol[:, z, :]
        else: return vol[:, :, z]

    def _extract_stack(vol, axis, z):
        if axis == 0: return vol[z-wr : z+wr+1]
        elif axis == 1: return vol[:, z-wr : z+wr+1, :].transpose(1, 0, 2)
        else: return vol[:, :, z-wr : z+wr+1].transpose(2, 0, 1)

    while True:
        axis, D, _, (H, W) = ORIENT_CONFIG[np.random.randint(3)]

        mr_subj_idx = np.random.randint(0, n_subj, size=batch_size)
        ct_subj_idx = np.random.randint(0, n_subj, size=batch_size)
        for k in range(batch_size):
            while ct_subj_idx[k] == mr_subj_idx[k]:
                ct_subj_idx[k] = np.random.randint(0, n_subj)

        z_idx = np.random.randint(wr, D - wr, size=batch_size)

        mr_batch, ct_batch = [], []
        mr_seg_batch, ct_seg_batch = [], []
        for sm, sc, z in zip(mr_subj_idx, ct_subj_idx, z_idx):
            mr_batch.append(np.ascontiguousarray(_extract_stack(mr_vols[sm], axis, z)))
            ct_batch.append(np.ascontiguousarray(_extract_stack(ct_vols[sc], axis, z)))
            mr_seg_batch.append(_extract_slice(mr_segs[sm], axis, z))
            ct_seg_batch.append(_extract_slice(ct_segs[sc], axis, z))

        # PyTorch format: (B, C, H, W)
        mr_stack = np.stack(mr_batch).astype(np.float32)  # (B, H, W, 7)
        ct_stack = np.stack(ct_batch).astype(np.float32)
        mr_seg = np.stack(mr_seg_batch).astype(np.int64)  # (B, H, W)
        ct_seg = np.stack(ct_seg_batch).astype(np.int64)

        yield (mr_stack, ct_stack, mr_seg, ct_seg)

# Test generator
test_gen = multi_orient_generator(train_mr_vols, train_ct_vols,
                                   train_mr_segs, train_ct_segs, batch_size=2)
mr, ct, mrs, cts = next(test_gen)
print(f"Batch shapes: mr={mr.shape}, ct={ct.shape}, mr_seg={mrs.shape}, ct_seg={cts.shape}")
del test_gen, mr, ct, mrs, cts


Batch shapes: mr=(2, 7, 176, 192), ct=(2, 7, 176, 192), mr_seg=(2, 176, 192), ct_seg=(2, 176, 192)


In [4]:
class Vxm2p5dDenseCore(nn.Module):
    def __init__(self, n_stack, enc_feats=(16, 32, 32, 32), final_feats=(32, 16), flow_scale=0.1):
        super().__init__()
        
        self.enc_blocks = nn.ModuleList()
        self.down_blocks = nn.ModuleList()
        in_ch = n_stack * 2
        
        for nf in enc_feats:
            self.enc_blocks.append(nn.Sequential(
                nn.Conv2d(in_ch, nf, 3, padding=1, bias=False),
                nn.BatchNorm2d(nf),
                nn.LeakyReLU(0.1)
            ))
            self.down_blocks.append(nn.Sequential(
                nn.Conv2d(nf, nf, 3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(nf),
                nn.LeakyReLU(0.1)
            ))
            in_ch = nf
            
        self.bottleneck = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1)
        )
        
        self.up_blocks = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        
        in_ch = 32
        for skip_ch in reversed(enc_feats):
            self.up_blocks.append(nn.Upsample(scale_factor=2, mode='nearest'))
            self.dec_blocks.append(nn.Sequential(
                nn.Conv2d(in_ch + skip_ch, skip_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(skip_ch),
                nn.LeakyReLU(0.1)
            ))
            in_ch = skip_ch
            
        self.final_conv0 = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.1)
        )
        self.final_conv1 = nn.Sequential(
            nn.Conv2d(32, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.LeakyReLU(0.1)
        )
        
        self.flow_unscaled = nn.Conv2d(16, 2, 3, padding=1)
        nn.init.zeros_(self.flow_unscaled.weight)
        nn.init.zeros_(self.flow_unscaled.bias)
        
        self.flow_scale = nn.Conv2d(2, 2, 1, bias=False)
        self.flow_scale.weight.data.zero_()
        self.flow_scale.weight.data[0, 0, 0, 0] = flow_scale
        self.flow_scale.weight.data[1, 1, 0, 0] = flow_scale
        self.flow_scale.weight.requires_grad = False
        
    def forward(self, moving_stack, fixed_stack):
        x = torch.cat([moving_stack, fixed_stack], dim=1)
        skips = []
        for enc, ds in zip(self.enc_blocks, self.down_blocks):
            x = enc(x)
            skips.append(x)
            x = ds(x)
            
        x = self.bottleneck(x)
        
        for up, dec, skip in zip(self.up_blocks, self.dec_blocks, reversed(skips)):
            x = up(x)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
            
        x = self.final_conv0(x)
        x = self.final_conv1(x)
        
        flow = self.flow_unscaled(x)
        flow_scaled = self.flow_scale(flow)
        return flow_scaled

model = Vxm2p5dDenseCore(n_stack=7, flow_scale=0.1).to(device)
print(model)


Vxm2p5dDenseCore(
  (enc_blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(14, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.1)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.1)
    )
    (2-3): 2 x Sequential(
      (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.1)
    )
  )
  (down_blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=Tr

In [5]:
def spatial_transform_2d(src, flow):
    # src: (B, C, H, W)
    # flow: (B, 2, H, W)
    b, c, h, w = src.shape
    
    vectors = [torch.arange(0, h), torch.arange(0, w)]
    grids = torch.meshgrid(vectors, indexing='ij')
    y_grid = grids[0].to(src.device).float()
    x_grid = grids[1].to(src.device).float()
    
    grid = torch.stack([x_grid, y_grid], dim=0).unsqueeze(0).expand(b, -1, -1, -1)
    new_locs = grid + flow
    
    shape = torch.tensor([w - 1, h - 1], device=src.device).float().view(1, 2, 1, 1)
    new_locs = (new_locs / shape) * 2.0 - 1.0
    new_locs = new_locs.permute(0, 2, 3, 1)
    
    return F.grid_sample(src, new_locs, align_corners=True, mode='bilinear', padding_mode='border')

def seg_to_onehot(seg, labels=SEG_LABELS, device=device):
    # seg: (B, H, W)
    oh = [seg == lbl for lbl in labels]
    oh = torch.stack(oh, dim=1).float() # (B, C, H, W)
    return oh.to(device)


In [6]:
BATCH_SIZE = 8
STEPS_PER_EPOCH = 200
VAL_STEPS = 50
EPOCHS = 800
SMOOTH_WEIGHT = 0.1
MI_WEIGHT = 0.5
CKPT_EVERY = 100

SAVE_DIR = '../trained_weights'
CKPT_DIR = os.path.join(SAVE_DIR, '2p5d_pt_checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(SAVE_DIR, '2p5d_dense_pt_best.pth')
FINAL_MODEL_PATH = os.path.join(SAVE_DIR, '2p5d_dense_pt.pth')

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_gen = multi_orient_generator(train_mr_vols, train_ct_vols, train_mr_segs, train_ct_segs, batch_size=BATCH_SIZE)
val_batches = [next(multi_orient_generator(val_mr_vols, val_ct_vols, val_mr_segs, val_ct_segs, batch_size=BATCH_SIZE)) 
               for _ in range(VAL_STEPS)]
               
train_history, val_history = [], []
best_loss = float('inf')


In [7]:
def flow_gradient_loss_2d(flow):
    dy = torch.abs(flow[:, :, 1:, :] - flow[:, :, :-1, :])
    dx = torch.abs(flow[:, :, :, 1:] - flow[:, :, :, :-1])
    return torch.mean(dx * dx) + torch.mean(dy * dy)

def dice_loss(y_pred, y_true):
    inter = torch.sum(y_pred * y_true, dim=(0, 2, 3))
    union = torch.sum(y_pred, dim=(0, 2, 3)) + torch.sum(y_true, dim=(0, 2, 3))
    return 1.0 - torch.mean((2.0 * inter + 1e-5) / (union + 1e-5))

class PatchwiseMI(nn.Module):
    def __init__(self, num_bins=32, sigma_ratio=0.5, max_clip=1.0, device='cuda'):
        super().__init__()
        self.max_clip = max_clip
        self.num_bins = num_bins
        self.bin_centers = torch.linspace(0, max_clip, num_bins, device=device)
        sigma = (max_clip / (num_bins - 1)) * sigma_ratio
        self.preterm = 1.0 / (2 * (sigma ** 2))

    def _soft_encoding(self, vol):
        vol = torch.clamp(vol, 0.0, self.max_clip).unsqueeze(-1)
        diff = vol - self.bin_centers.view(1, 1, 1, 1, -1)
        enc = torch.exp(-self.preterm * diff ** 2)
        enc = enc / (enc.sum(dim=-1, keepdim=True) + 1e-5)
        return enc

    def forward(self, I, J):
        I = I.float()
        J = J.float()
        B = I.shape[0]
        I_enc = self._soft_encoding(I)
        J_enc = self._soft_encoding(J)
        
        I_enc = I_enc.view(B, -1, self.num_bins)
        J_enc = J_enc.view(B, -1, self.num_bins)
        
        I_perm = I_enc.permute(0, 2, 1)
        pab = torch.bmm(I_perm, J_enc) / I_enc.shape[1]
        
        pa = I_enc.mean(dim=1, keepdim=True)
        pb = J_enc.mean(dim=1, keepdim=True)
        papb = torch.bmm(pa.permute(0, 2, 1), pb) + 1e-5
        
        mi = (pab * torch.log((pab + 1e-5) / papb)).sum(dim=(1, 2))
        return -mi.mean()

patchwise_mi = PatchwiseMI(device=device)

def train_step(mr_stack, ct_stack, mr_seg, ct_seg):
    optimizer.zero_grad()
    
    flow = model(mr_stack, ct_stack)
    smooth_loss = flow_gradient_loss_2d(flow)
        
    mr_seg_oh = seg_to_onehot(mr_seg)
    ct_seg_oh = seg_to_onehot(ct_seg)
    warped_seg_oh = spatial_transform_2d(mr_seg_oh, flow)
    d_loss = dice_loss(warped_seg_oh, ct_seg_oh)
        
    mr_center = mr_stack[:, WINDOW_RADIUS:WINDOW_RADIUS+1, ...]
    ct_center = ct_stack[:, WINDOW_RADIUS:WINDOW_RADIUS+1, ...]
    warped_mr_center = spatial_transform_2d(mr_center, flow)
    mi_loss_val = patchwise_mi(warped_mr_center, ct_center)
        
    total_loss = d_loss + MI_WEIGHT * mi_loss_val + SMOOTH_WEIGHT * smooth_loss
        
    total_loss.backward()
    optimizer.step()
    
    return total_loss.item()

def val_step(mr_stack, ct_stack, mr_seg, ct_seg):
    with torch.no_grad():
            flow = model(mr_stack, ct_stack)
            smooth_loss = flow_gradient_loss_2d(flow)
            
            mr_seg_oh = seg_to_onehot(mr_seg)
            ct_seg_oh = seg_to_onehot(ct_seg)
            warped_seg_oh = spatial_transform_2d(mr_seg_oh, flow)
            d_loss = dice_loss(warped_seg_oh, ct_seg_oh)
            
            mr_center = mr_stack[:, WINDOW_RADIUS:WINDOW_RADIUS+1, ...]
            ct_center = ct_stack[:, WINDOW_RADIUS:WINDOW_RADIUS+1, ...]
            warped_mr_center = spatial_transform_2d(mr_center, flow)
            mi_loss_val = patchwise_mi(warped_mr_center, ct_center)
            
            total_loss = d_loss + MI_WEIGHT * mi_loss_val + SMOOTH_WEIGHT * smooth_loss
    return total_loss.item()


In [ ]:
for epoch in range(EPOCHS):
    model.train()
    running_train = 0.0
    nan_steps = 0
    
    for step in range(STEPS_PER_EPOCH):
        mr_stack_np, ct_stack_np, mr_seg_np, ct_seg_np = next(train_gen)
        
        mr_stack = torch.from_numpy(mr_stack_np).to(device)
        ct_stack = torch.from_numpy(ct_stack_np).to(device)
        mr_seg = torch.from_numpy(mr_seg_np).to(device)
        ct_seg = torch.from_numpy(ct_seg_np).to(device)
        
        loss = train_step(mr_stack, ct_stack, mr_seg, ct_seg)
        
        if np.isfinite(loss):
            running_train += loss
        else:
            nan_steps += 1
            
    valid_steps = STEPS_PER_EPOCH - nan_steps
    epoch_train = running_train / max(valid_steps, 1)
    train_history.append(epoch_train)
    
    model.eval()
    running_val = 0.0
    for mr_v, ct_v, mrs_v, cts_v in val_batches:
        mr_stack = torch.from_numpy(mr_v).to(device)
        ct_stack = torch.from_numpy(ct_v).to(device)
        mr_seg = torch.from_numpy(mrs_v).to(device)
        ct_seg = torch.from_numpy(cts_v).to(device)
        
        loss_v = val_step(mr_stack, ct_stack, mr_seg, ct_seg)
        running_val += loss_v
        
    epoch_val = running_val / len(val_batches)
    val_history.append(epoch_val)
    
    nan_str = f'  ({nan_steps} NaN)' if nan_steps else ''
    print(f'Epoch {epoch+1}/{EPOCHS} -- train: {epoch_train:.6f} | val: {epoch_val:.6f}{nan_str}')
    
    if (epoch + 1) % CKPT_EVERY == 0:
        ckpt_path = os.path.join(CKPT_DIR, f'2p5d_pt_epoch_{epoch+1}.pth')
        torch.save(model.state_dict(), ckpt_path)
        
    if epoch_val < best_loss:
        best_loss = epoch_val
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        
torch.save(model.state_dict(), FINAL_MODEL_PATH)
print(f'\nBest val loss: {best_loss:.6f}')
print('Training complete!')


Epoch 1/800 -- train: 0.163291 | val: 0.189552
Epoch 2/800 -- train: 0.162579 | val: 0.185322
Epoch 3/800 -- train: 0.160805 | val: 0.183947
Epoch 4/800 -- train: 0.161986 | val: 0.178442
Epoch 5/800 -- train: 0.157123 | val: 0.176082
Epoch 6/800 -- train: 0.159368 | val: 0.169236
Epoch 7/800 -- train: 0.156977 | val: 0.162892
Epoch 8/800 -- train: 0.144114 | val: 0.159699
Epoch 9/800 -- train: 0.151579 | val: 0.159993
Epoch 10/800 -- train: 0.158057 | val: 0.150975
Epoch 11/800 -- train: 0.143131 | val: 0.152717
Epoch 12/800 -- train: 0.136394 | val: 0.142968
Epoch 13/800 -- train: 0.142242 | val: 0.150632
Epoch 14/800 -- train: 0.134324 | val: 0.148718
Epoch 15/800 -- train: 0.133016 | val: 0.140193
Epoch 16/800 -- train: 0.133284 | val: 0.131372
Epoch 17/800 -- train: 0.128291 | val: 0.140976
Epoch 18/800 -- train: 0.131339 | val: 0.126588
Epoch 19/800 -- train: 0.125445 | val: 0.122954
Epoch 20/800 -- train: 0.114956 | val: 0.124410
Epoch 21/800 -- train: 0.112513 | val: 0.119480
E

In [ ]:
epochs_ran = np.arange(1, len(train_history) + 1)
plt.figure(figsize=(10, 6))
plt.plot(epochs_ran, train_history, '.-', label='train')
plt.plot(epochs_ran, val_history,   '.-', label='val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('2.5D TensorFlow Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()